# LLM Structured Output — From Free Text to Typed Data

> **Description:** This notebook builds up *structured output* step by step, starting from a plain LLM call that returns prose and ending with validated, typed Python objects you can use directly in code.

By default, an LLM answers in natural language. That is great for a chat window and terrible for a program: prose cannot be indexed, filtered, stored in a database column, or passed to the next function without someone first parsing it back out. **Structured output** is the set of techniques that make the model's response predictable enough for code to consume — first as loosely-shaped JSON, then as JSON that matches an exact schema, and finally as a real Python object with real types.

This matters because it is the difference between a demo and a production system: a support-ticket triage step, a form-filling agent, or a data-extraction pipeline is only as reliable as the shape of the data flowing between its steps.

## Setup

Load `OPENAI_API_KEY` from `.env` and create a single reusable OpenAI client.

> Run this notebook from the `part_2_concepts/` folder — that's what makes the relative `.env` path work, matching the other notebooks here.

In [13]:
import os
from dotenv import load_dotenv


load_dotenv(".env", override=True)

print("OPENAI_API_KEY set:", bool(os.environ.get("OPENAI_API_KEY")))

OPENAI_API_KEY set: True


## 1. Baseline: A Plain LLM Call Returns Prose

No `response_format`, no schema, no instructions about shape — just a normal `chat.completions.create()` call. The model reads the text and answers in whatever words it thinks read best.

The task below asks it to pull three pieces of information out of a sentence. A human can read the answer easily. Code cannot: there is no `["age"]` to index into, only a string.

In [14]:
from openai import OpenAI

client = OpenAI()
model = "gpt-5.4-nano"
text = "Maria is a 29-year-old data engineer who just relocated to Austin."

response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "user",
            "content": f"Extract the person's name, age, and city from this text:\n\n{text}",
        }
    ],
)

raw_answer = response.choices[0].message.content
print(raw_answer)
print()
print("type:", type(raw_answer))

- **Name:** Maria  
- **Age:** 29  
- **City:** Austin

type: <class 'str'>


The exact wording of `raw_answer` is not guaranteed — it might be a sentence, a bulleted list, or something else entirely, and it can change between calls to the same prompt. Anything downstream that needs `age` as a number has to write a custom parser for whatever shape shows up today, and that parser breaks the next time the model phrases things differently.

## 2. The Naive Fix: Asking Nicely for JSON

The obvious first move is to just ask for JSON in the prompt. This often works — but "often" is doing a lot of work in that sentence. Nothing stops the model from:

- wrapping the JSON in a ` ```json ... ``` ` markdown fence,
- adding a sentence of commentary before or after it,
- renaming a field, or changing `29` to `"29"`.

Every one of those breaks a plain `json.loads()` call. This is a prompting convention, not a contract the API enforces.

In [15]:
naive_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "user",
            "content": (
                "Extract the person's name, age, and city from this text as a JSON object "
                f"with keys 'name', 'age', 'city'.\n\n{text}"
            ),
        }
    ],
)

naive_content = naive_response.choices[0].message.content
print(naive_content)

```json
{
  "name": "Maria",
  "age": 29,
  "city": "Austin"
}
```


In [16]:
import json

# Parsing it back out means hoping the model behaved:
try:
    parsed = json.loads(naive_content)
    print("Parsed OK:", parsed)
except json.JSONDecodeError as exc:
    print("json.loads failed:", exc)
    print("This is the exact failure mode 'please respond in JSON' cannot prevent.")

json.loads failed: Expecting value: line 1 column 1 (char 0)
This is the exact failure mode 'please respond in JSON' cannot prevent.


This is usually where teams reach for defensive string-stripping (`content.strip("`").removeprefix("json")`, regexing out the fenced block, and so on). Every one of those is a patch on top of a prompt that never had a real guarantee behind it. The next three sections replace "please" with API-level enforcement.

## 3. JSON Mode: `response_format={"type": "json_object"}`

Passing `response_format={"type": "json_object"}` tells the API itself — not just the prompt — to only emit syntactically valid JSON. No markdown fences, no leading "Sure, here you go:", no chance of a `JSONDecodeError`.

It does **not** guarantee which keys or types you get — the model still decides the schema, so you still have to describe it in the prompt. And OpenAI requires the literal word "json" to appear somewhere in the messages when this mode is on, or the request is rejected.

In [30]:
json_mode_response = client.chat.completions.create(
    model=model,
    response_format={"type": "json_object"},
    messages=[
        {
            "role": "system",
            "content": "Extract structured data as JSON with keys: name, age, city.",
        },
        {"role": "user", "content": text},
    ],
)

data = json.loads(json_mode_response.choices[0].message.content)
print(data)
print({key: type(value).__name__ for key, value in data.items()})

{'name': 'Maria', 'age': 29, 'city': 'Austin'}
{'name': 'str', 'age': 'int', 'city': 'str'}


`json.loads()` is now guaranteed to succeed. But run this a few times and watch the value types: `age` might come back as the integer `29` or the string `"29"` depending on the model's mood, because nothing pins the schema down. JSON mode solves *syntax*, not *shape*.

## 4. Structured Outputs: Enforcing a JSON Schema

OpenAI's **Structured Outputs** feature (`response_format={"type": "json_schema", ...}` with `"strict": true`) goes one step further: the API constrains generation so the response matches your JSON Schema exactly — every required key present, every type correct, no extra keys sneaking in. This is enforced during decoding, not requested in the prompt.

In [18]:
person_schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "person_info",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "name": {"type": "string"},
                "age": {"type": "integer"},
                "city": {"type": "string"},
            },
            "required": ["name", "age", "city"],
            "additionalProperties": False,
        },
    },
}

schema_response = client.chat.completions.create(
    model=model,
    response_format=person_schema,
    messages=[
        {"role": "user", "content": f"Extract the person's info from: {text}"}
    ],
)

person_data = json.loads(schema_response.choices[0].message.content)
print(person_data)
print("age is a real int:", isinstance(person_data["age"], int))

{'name': 'Maria', 'age': 29, 'city': 'Austin'}
age is a real int: True


This works, and it is bulletproof — but hand-writing JSON Schema for every shape you need is verbose, and it is easy for the schema to quietly drift out of sync with the Python code that consumes it. Section 5 removes that duplication.

## 5. The Ergonomic Path: Pydantic + `client.chat.completions.parse()`

Define the shape once, as a normal Pydantic model. `client.chat.completions.parse()` derives the JSON Schema from the model automatically, sends it with `strict` mode under the hood, validates the response, and hands back a real Python object on `.choices[0].message.parsed` — no `json.loads()` in sight.

In [33]:
from pydantic import BaseModel

class PersonInfo(BaseModel):
    name: str
    age: int
    city: str


completion = client.chat.completions.parse(
    model=model,
    messages=[
        {"role": "user", "content": f"Extract the person's info from: {text}"}
    ],
    response_format=PersonInfo,
)

person = completion.choices[0].message.parsed
print(person)


name='Maria' age=29 city='Austin'


`person` is a `PersonInfo` instance. Pydantic already validated it, your editor already knows `person.age` is an `int`, and if the model somehow returned something that didn't fit the schema, `.parse()` would raise instead of handing you silently-wrong data. This is the pattern to reach for by default.

## 6. Nested Models and Lists

Real payloads are rarely flat. Pydantic models compose the same way they would anywhere else in Python — nest a `BaseModel` inside another, or wrap one in a `list[...]`, and `.parse()` handles the extra structure automatically.

In [34]:
class Experience(BaseModel):
    title: str
    company: str
    years: int


class Resume(BaseModel):
    name: str
    email: str
    skills: list[str]
    experience: list[Experience]


resume_text = (
    "Jordan Lee (jordan.lee@example.com) has spent 3 years as a Backend Engineer at Nimbus Cloud "
    "and 2 years as a Software Engineer at Delta Systems, and lists Python, SQL, and Kubernetes as skills."
)

resume_completion = client.chat.completions.parse(
    model=model,
    messages=[
        {"role": "user", "content": f"Extract a structured resume from:\n\n{resume_text}"}
    ],
    response_format=Resume,
)

resume = resume_completion.choices[0].message.parsed
print(resume.model_dump_json(indent=2))

{
  "name": "Jordan Lee",
  "email": "jordan.lee@example.com",
  "skills": [
    "Python",
    "SQL",
    "Kubernetes"
  ],
  "experience": [
    {
      "title": "Backend Engineer",
      "company": "Nimbus Cloud",
      "years": 3
    },
    {
      "title": "Software Engineer",
      "company": "Delta Systems",
      "years": 2
    }
  ]
}


`resume.experience` is a `list[Experience]` — real objects, not dictionaries you have to hope are shaped right. `resume.experience[0].years` is already an `int` you can sum or sort on.

## 7. Enums, Optional Fields, and Refusals

`Enum` fields constrain the model to one of a fixed set of values — useful any time you need a category, a priority, or a status rather than free text. Pydantic's `Optional[...]` / `| None` marks a field the model can leave empty instead of inventing a value.

One more thing worth handling explicitly: the model can refuse to fill in a schema (for example, if the input asks for something unsafe). Structured Outputs surfaces that as `message.refusal` instead of quietly returning garbage, so always check it before trusting `.parsed`.

In [ ]:
from enum import Enum
from pydantic import Field


class Category(str, Enum):
    BILLING = "billing"
    TECHNICAL = "technical"
    ACCOUNT = "account"
    OTHER = "other"


class Priority(str, Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"


class SupportTicket(BaseModel):
    category: Category
    priority: Priority
    summary: str = Field(description="One sentence describing the issue.")
    needs_human: bool


ticket_email = (
    "Hi, I was charged twice for my subscription this month and I need this fixed today, "
    "it's really frustrating."
)

ticket_completion = client.chat.completions.parse(
    model=model,
    messages=[
        {
            "role": "system",
            "content": "Triage the customer email into a support ticket.",
        },
        {"role": "user", "content": ticket_email},
    ],
    response_format=SupportTicket,
)

ticket_message = ticket_completion.choices[0].message

if ticket_message.refusal:
    print("Model refused:", ticket_message.refusal)
else:
    ticket = ticket_message.parsed
    print(ticket)

category=<Category.BILLING: 'billing'> priority=<Priority.HIGH: 'high'> summary='Customer reports being charged twice for their subscription this month and requests the issue be fixed today.' needs_human=True


`ticket.category` and `ticket.priority` can only ever be one of the values defined on the `Enum` — there is no `"urgent"` sneaking in next to `"high"` because someone phrased an email differently. That guarantee is what makes it safe to write `if ticket.priority == Priority.HIGH:` downstream without also writing a normalization layer.

## 8. Putting It Together: Batch Triage

This is where structured output stops being a syntax trick and starts being genuinely useful. Feed a batch of unrelated, messy customer emails through the same `SupportTicket` schema, and the result is a list of real Python objects you can filter, sort, and aggregate — something that is simply not possible with a batch of free-text replies.

In [35]:
inbox = [
    "My app crashes every time I try to export a report. Please help soon.",
    "Just wanted to say the new dashboard redesign looks great, nice work!",
    "I can't log into my account anymore, it says my password is wrong even after I reset it.",
    "Can you update the billing address on my account to 42 Market St?",
]

tickets: list[SupportTicket] = []

for email_text in inbox:
    result = client.chat.completions.parse(
        model=model,
        messages=[
            {"role": "system", "content": "Triage the customer email into a support ticket."},
            {"role": "user", "content": email_text},
        ],
        response_format=SupportTicket,
    )
    message = result.choices[0].message
    if not message.refusal:
        tickets.append(message.parsed)

for ticket in tickets:
    print(f"[{ticket.priority.value}] {ticket.category.value} needs_human={ticket.needs_human}  {ticket.summary}")

[high] technical needs_human=True  The app crashes whenever the user attempts to export a report.
[low] other needs_human=False  Customer provided positive feedback about the new dashboard redesign.
[high] account needs_human=True  Customer cannot log in; system reports incorrect password even after a password reset.
[medium] billing needs_human=True  Customer requests updating the billing address on their account to 42 Market St.


In [23]:
high_priority = [t for t in tickets if t.priority == Priority.HIGH]
print(f"{len(high_priority)} of {len(tickets)} tickets need urgent attention")

by_category: dict[str, int] = {}
for ticket in tickets:
    by_category[ticket.category.value] = by_category.get(ticket.category.value, 0) + 1
print("Volume by category:", by_category)

2 of 4 tickets need urgent attention
Volume by category: {'technical': 1, 'other': 1, 'account': 1, 'billing': 1}


Filtering on `.priority == Priority.HIGH` and counting by `.category.value` is ordinary Python — no regex, no fuzzy string matching, no hoping the model spelled "billing" the same way twice. That reliability is the entire point of structured output.

## Summary

| Technique | Guarantees | Effort |
|---|---|---|
| Plain call, no `response_format` | Nothing — free-text prose | None |
| "Please reply in JSON" (prompt only) | Nothing enforced by the API | Low, but fragile |
| `response_format={"type": "json_object"}` | Valid JSON syntax | Low |
| `response_format={"type": "json_schema", ...}` | Valid JSON **and** matches your schema exactly | Medium — hand-written schema |
| Pydantic model + `client.chat.completions.parse()` | Same guarantee as above, plus a real typed Python object on `.parsed` | Low — schema comes from your types |

**Rule of thumb:** default to a Pydantic model with `.parse()`. Reach for raw `json_schema` only when the schema has to be built dynamically or shared with a non-Python consumer, and reach for JSON mode only when you cannot use `.parse()` at all.

**See also:** `tool_calling.ipynb` in this folder covers the related but different problem of letting the model *take actions* (calling your functions) rather than just *shaping its final answer*. The two combine naturally: a tool's arguments are already validated with this exact schema mechanism.